In [ ]:
using LensFactory
using LensFactory.Constants
using CairoMakie

# Initialize a cosmology
The observer, lens, and source combined together create what we call a *lens system*. The first step in doing lensing is to define a `Cosmology`. Below we initialize the default cosmology and assume that our lens and source are at ``z=0.5`` and ``z=1.5``, respectively.

In [ ]:
# Initialize default cosmology
cosmo = Cosmology.init_cosmology()

# Lens and source redshifts
zl = 0.5
zs = 1.5

# Angular diameter distances and distance ratio
Dol = Cosmology.angular_diameter_distance(cosmo, 0., zl)
Dls = Cosmology.angular_diameter_distance(cosmo, zl, zs)
Dos = Cosmology.angular_diameter_distance(cosmo, 0., zs)
adis = Dls/Dos

# Create a grid
The next step is to create a grid that would represent the sky in which we see the (strong) lensing. [`Lenses`](https://akmeena766.github.io/LensFactory.jl/stable/Lenses) module has a function called [`get_meshgrid`]((https://akmeena766.github.io/LensFactory.jl/stable/Lenses/#LensFactory.Lenses.get_meshgrid)) that creates a square-pixel grid given half-size along x- and y-axis with the pixel resolution.

Since in most of the astrophysical scenario (primarily strong lensing by galaxies or galaxy clusters), the Einstein angle is ~[0.1, 50] arcseconds, it was decided to have `ANGLE_ARCSECOND` as default unit for various calculations in `LensFactory`. Hence, when we create grid, it is created assuming `ANGLE_ARCSECOND` as the unit.

In [ ]:
# Create image plane grid (default units are ANGLE_ARCSEC)
x, y = Lenses.get_meshgrid(5, 5, 0.06);

# Initialize a point mass lens and plot
In `LensFactory` every lens profiles comes with an initializer struct and its name start with, ``init_``. For example, the initializer for point mass lens is called `init_PointLens`.

To initialize any lens, we need to know the number of parameters that it requires. For a point mass lens, we need to define its position (x_c, y_c), its mass M, and the distance to lens plane D_d. In `LensFactory`, all lenses are by default placed at origin. Hence, to initialize a point mass lens at the origin, we need to provide (D_d, M). **The lens mass should be provided solar mass units.**

In [ ]:
# Initialize an isolated point mass lens
lens = Lenses.init_PointLens(D_d=Dol, mass=1E12)

Now that we have initialized a point lens, the next step is to calculate its properties (potential, defelection, or magnification) at any given position. For that, we call various functions in [`Lenses`](https://akmeena766.github.io/LensFactory.jl/stable/Lenses) module. For example, let us calculate the deflection on the above-defined grid and use the in-built plotting functions to plot the components. In this cell, we first generate an empty sky using [`plot_sky`](https://akmeena766.github.io/LensFactory.jl/stable/PlotExt_Lenses/#LensFactory.Lenses.plot_sky) function and then plot the deflection maps on top of it.

In [ ]:
# Calculate deflection angle. The output values are scaled for a source at z = ∞ (i.e., D_ds/D_s=1).
dx, dy = Lenses.get_deflection(lens, x, y)

# Let's plot the deflection angles
fig, ax = Lenses.plot_sky(5, 5)
heatmap!(ax, x[:,1], y[1,:], dx, colorrange=(-5, 5))
display(fig)

fig, ax = Lenses.plot_sky(5, 5)
heatmap!(ax, x[:,1], y[1,:], dy, colorrange=(-5, 5))
display(fig)

`LensFactory` also has in-built function to plot the image plane, [`plot_image_plane`](https://akmeena766.github.io/LensFactory.jl/stable/PlotExt_Lenses/#LensFactory.Lenses.plot_image_plane) for a given source redshift. Below, we demonstrate its usage for point mass lens.

In [ ]:
# Plot the image plane (by default both source and image plane are shown in the same panel)
fig, ax = Lenses.plot_image_plane(lens, x, y, adis)
display(fig)

# Let us put a point source at (1", 1") in the source plane (the image at the center is a numerical 
# artifact due to singular nature of the point lens profile).
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; source=(1, 1))
display(fig)

# Point Mass Lens + External Shear
For multi-component lens models, made of different type of lenses, we use composite lens model. For example, to simulate point mass lens in environment effect, we can create a composite lens with two components, `PointLens` and `ExternalEffectsLens`. Now we need to explicitly provide every parameters for each lens component.

**While using, the `ExternalEffects` lens, it is important to note that we should pass ($\kappa$, $\gamma$) values for a source at infinity (i.e., $D_{ds}/D_s$). This is because all lensing qunatities are later re-scaled by `adis` for a given source redshift.**

In [ ]:
lens = Lenses.init_CompositeLens([(lens=:PointLens, x_c=0.0, y_c=0.0, D_d=Dol, mass=1E12),
                                  (lens=:ExternalEffects, kappa=0.0, gamma=0.1, angle=45)])

# Plot the image plane (again the image at the center is a numerical artifact).
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; source=(0.1, 0.1))
display(fig)

# Two point mass lenses
Now let us construct a lens made of two point mass lenses. We again use the composite lens and provide parameters for each lens component.

In [ ]:
# Initialize a two component lens
lens = Lenses.init_CompositeLens([(lens=:PointLens, D_d=Dol, x_c=0.0, y_c=0.0, mass=1E11),
                                  (lens=:PointLens, D_d=Dol, x_c=1.0, y_c=1.0, mass=1E11)])

# Plot the image plane (by default both source and image plane are shown in the same panel)
fig, ax = Lenses.plot_image_plane(lens, x, y, adis)
display(fig)

# N-point mass lens
The same composite lens can also be used to mimic a ensamble of N point mass lenses. Someone familiar with lensing can see the resemblance with microlensing. **However, it is important to note that this default method is not optimized for microlensing by a bunch of point mass lenses.**

In [ ]:
# To control the positions of point mass lenses
using Random
Random.seed!(1234)

# Initialize a lens made of multiple point mass lenses
n_point = 15
ensamble = [(lens=:PointLens, D_d=Dol, x_c=(-3.0 + 6.0*rand()), y_c = (-3.0 + 6.0*rand()), mass=1E11) for _ in 1:n_point]
lens = Lenses.init_CompositeLens(ensamble)

# Here we have split the source and image plane for better visualization using "two_panel" keyword
fig, ax = Lenses.plot_image_plane(lens, x, y, adis; two_panel=true, figure_size=(800, 400))
display(fig)
